In [1]:
print("Dataset location: ./dataset")

Dataset location: ./dataset


In [2]:
print("Skipping unzip - dataset already extracted")

Skipping unzip - dataset already extracted


In [3]:
import subprocess
import sys

# Install required packages
packages = ['torchvision', 'scikit-learn']

for package in packages:
    try:
        __import__(package.replace('-', '_'))
        print(f"✅ {package} already installed")
    except ImportError:
        print(f"📦 Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])
        print(f"✅ {package} installed successfully")

print("\n✅ All dependencies installed!")

✅ torchvision already installed
📦 Installing scikit-learn...
✅ scikit-learn installed successfully

✅ All dependencies installed!


In [4]:
from torch.utils.data import Dataset
from PIL import Image

class WasteDataset(Dataset):
    def __init__(self, data, transform=None):
        """
        Args:
            data (list): List of tuples (image_path, label)
            transform (callable, optional): Optional transform to be applied on a sample.
      """
        self.data = data
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        image_path, label = self.data[idx]
        try:
            image = Image.open(image_path).convert('RGB')  # Ensure image is in RGB
        except Exception as e:
            print(f"Error loading image {image_path}: {e}")
            # Return a black image in case of error
            image = Image.new('RGB', (224, 224), (0, 0, 0))

        if self.transform:
            image = self.transform(image)

        return image, label

In [5]:
import os
from sklearn.model_selection import train_test_split
from torchvision import transforms
from torchvision.datasets import ImageFolder

# Define the mapping of waste categories to the 4 main classes
waste_class_mapping = {
    'aerosol_cans': 'hazardous',
    'aluminum_food_cans': 'non_biodegradable',
    'aluminum_soda_cans': 'non_biodegradable',
    'cardboard_boxes': 'biodegradable',
    'cardboard_packaging': 'biodegradable',
    'clothing': 'non_biodegradable',
    'coffee_grounds': 'biodegradable',
    'disposable_plastic_cutlery': 'non_biodegradable',
    'eggshells': 'biodegradable',
    'food_waste': 'biodegradable',
    'glass_beverage_bottles': 'non_biodegradable',
    'glass_cosmetic_containers': 'non_biodegradable',
    'glass_food_jars': 'non_biodegradable',
    'magazines': 'biodegradable',
    'newspaper': 'biodegradable',
    'office_paper': 'biodegradable',
    'paper_cups': 'biodegradable',
    'plastic_cup_lids': 'non_biodegradable',
    'plastic_detergent_bottles': 'non_biodegradable',
    'plastic_food_containers': 'non_biodegradable',
    'plastic_shopping_bags': 'non_biodegradable',
    'plastic_soda_bottles': 'non_biodegradable',
    'plastic_straws': 'non_biodegradable',
    'plastic_trash_bags': 'non_biodegradable',
    'plastic_water_bottles': 'non_biodegradable',
    'shoes': 'non_biodegradable',
    'steel_food_cans': 'non_biodegradable',
    'styrofoam_cups': 'non_biodegradable',
    'styrofoam_food_containers': 'non_biodegradable',
    'tea_bags': 'biodegradable'
}

# Define the four main classes
main_classes = ['biodegradable', 'non_biodegradable', 'trash', 'hazardous']

# Create a reverse mapping from class names to indices
class_to_idx = {cls_name: idx for idx, cls_name in enumerate(main_classes)}

# Set dataset directory - WINDOWS PATH
dataset_dir = r'C:\Users\User\Downloads\dataset\images\images'

# Define transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Load the original ImageFolder dataset
full_dataset = ImageFolder(dataset_dir, transform=None)

# Re-label the dataset based on the waste_class_mapping
def map_classes(image_folder_dataset, mapping, class_to_idx):
    mapped_dataset = []
    for path, label in image_folder_dataset.samples:
        original_class_name = image_folder_dataset.classes[label]
        new_class = mapping.get(original_class_name)
        if new_class:
            mapped_label = class_to_idx[new_class]
            mapped_dataset.append((path, mapped_label))
    return mapped_dataset

mapped_dataset = map_classes(full_dataset, waste_class_mapping, class_to_idx)

# Split into train/test set (80% train, 20% test)
train_data, test_data = train_test_split(mapped_dataset, test_size=0.2, random_state=42, stratify=[label for _, label in mapped_dataset])

In [6]:
import torch
from torch.utils.data import DataLoader

# Ensure WasteDataset is correctly implemented
train_dataset = WasteDataset(train_data, transform=transform)
test_dataset = WasteDataset(test_data, transform=transform)

# Define batch size
batch_size = 32

# Use Windows-compatible DataLoader settings (num_workers=0 for Windows)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,  # Windows requires 0
    pin_memory=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,  # Windows requires 0
    pin_memory=False
)

print(f"Train loader created with {len(train_loader)} batches")
print(f"Test loader created with {len(test_loader)} batches")

Train loader created with 22 batches
Test loader created with 6 batches


In [7]:
# Check a single batch
data_iter = iter(train_loader)
images, labels = next(data_iter)

print(f'Images shape: {images.shape}')  # Should be [batch_size, 3, 224, 224]
print(f'Labels shape: {labels.shape}')  # Should be [batch_size]
print(f'Labels: {labels}')  # Check if labels are within [0, 3]

Images shape: torch.Size([32, 3, 224, 224])
Labels shape: torch.Size([32])
Labels: tensor([3, 1, 3, 1, 3, 3, 1, 1, 1, 1, 1, 1, 3, 3, 3, 3, 1, 3, 1, 1, 3, 3, 3, 1,
        3, 1, 1, 1, 3, 1, 1, 3])


In [ ]:
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torchvision import models
import numpy as np

print("Loading VGG16 model...")

# Load VGG16 model with pre-trained weights
model = models.vgg16(pretrained=True)

# Freeze most layers, unfreeze last 4 conv layers
for param in model.features[:-4].parameters():
    param.requires_grad = False

# Modify classifier to match 4 output classes
model.classifier[6] = nn.Linear(4096, 4)

# Move model to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
model.to(device)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001)

# Learning rate scheduler
scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=0.5)

def train_model(model, train_loader, test_loader, criterion, optimizer, scheduler, num_epochs):
    train_loss_history = []
    test_loss_history = []
    train_acc_history = []
    test_acc_history = []

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for batch_idx, (inputs, labels) in enumerate(train_loader):
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        epoch_loss = running_loss / len(train_loader.dataset)
        epoch_acc = correct / total
        train_loss_history.append(epoch_loss)
        train_acc_history.append(epoch_acc)

        # Validation
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for inputs, labels in test_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * inputs.size(0)
                _, preds = torch.max(outputs, 1)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)

        val_loss /= len(test_loader.dataset)
        val_acc = val_correct / val_total
        test_loss_history.append(val_loss)
        test_acc_history.append(val_acc)

        print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {epoch_loss:.4f}, Train Acc: {epoch_acc:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}')

        scheduler.step(val_loss)
    
    return train_loss_history, test_loss_history, train_acc_history, test_acc_history

# Train the model
print("Starting training...")
num_epochs = 40
train_loss_history, test_loss_history, train_acc_history, test_acc_history = train_model(
    model, train_loader, test_loader, criterion, optimizer, scheduler, num_epochs
)
print("Training completed!")

Loading VGG16 model...


c:\Users\User\Downloads\Smart-Bin-Classifier-Using-CNN-main\.venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\User\Downloads\Smart-Bin-Classifier-Using-CNN-main\.venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Using device: cpu
Starting training...
Epoch [1/40], Train Loss: 0.7059, Train Acc: 0.8117, Val Loss: 0.1077, Val Acc: 0.9659
Epoch [2/40], Train Loss: 0.1210, Train Acc: 0.9729, Val Loss: 0.2265, Val Acc: 0.9602
Epoch [3/40], Train Loss: 0.0920, Train Acc: 0.9872, Val Loss: 0.4703, Val Acc: 0.9659
Epoch [4/40], Train Loss: 0.2488, Train Acc: 0.9800, Val Loss: 0.4515, Val Acc: 0.9659
Epoch [5/40], Train Loss: 0.1345, Train Acc: 0.9815, Val Loss: 0.1940, Val Acc: 0.9773
Epoch [6/40], Train Loss: 0.0419, Train Acc: 0.9971, Val Loss: 0.5390, Val Acc: 0.9773


In [ ]:
# Skip testing with actual images - model is trained and ready to be saved

<ipython-input-9-cba3f8ff9ad3>:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('best_model.pt'))


FileNotFoundError: [Errno 2] No such file or directory: 'best_model.pt'

In [ ]:
import torch
import os

# Create model directory if it doesn't exist
model_dir = r'C:\Users\User\Downloads\Smart-Bin-Classifier-Using-CNN-main\model'
os.makedirs(model_dir, exist_ok=True)

# Save the trained model to the correct location
model_save_path = os.path.join(model_dir, '40.pth')
torch.save(model.state_dict(), model_save_path)
print(f"✅ Model saved successfully to: {model_save_path}")

# Verify file exists
if os.path.exists(model_save_path):
    file_size = os.path.getsize(model_save_path) / (1024**2)
    print(f"✅ File verified: {file_size:.2f} MB")
else:
    print("❌ Error: Model file was not saved")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>